# check — SC Linux

Validate source configuration, IPA CA trust, pinned GitHub Actions and syntax.

Use a working copy created by `make lab` for interactive execution. For interactive execution, edit `NOTEBOOK_ARGS` in the next cell. CI passes these arguments through `scripts/notebook.sh`.

## Locate the repository

Jupyter kernels may start in `notebooks/` or `build/notebooks/`. Use the repository root for all relative paths.

In [ ]:
from pathlib import Path as _SCPath
import os as _sc_os
_start = _SCPath.cwd().resolve()
_candidates = (_start, *_start.parents)
_project_root = next(
    (path for path in _candidates if (path / "config/build.json").is_file() and (path / "notebooks").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the SC Linux checkout")
_sc_os.chdir(_project_root)
print("Repository:", _project_root)

In [ ]:
NOTEBOOK_ARGS = globals().get('NOTEBOOK_ARGS', [])

## Setup

In [ ]:
"""Check source inputs; use --require-trust before building images."""

import argparse
import configparser
import ast
import json
import re
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "lib"))
from scos.core import check_tree, read_identity, verify_ca

### parse_mkosi_packages

In [ ]:
def parse_mkosi_packages(value: str) -> list[str]:
    """Reject package groups that mkosi would forward as one APT argument."""
    packages = [entry.strip() for entry in re.split(r"[,\n]", value) if entry.strip()]
    if not packages or any(len(entry.split()) != 1 for entry in packages):
        raise ValueError(
            "mkosi Packages= requires comma-separated names or one package per line"
        )
    return packages

### check_mkosi_packages

In [ ]:
def check_mkosi_packages() -> None:
    for image in ("server", "installer"):
        path = ROOT / "images" / image / "mkosi.conf"
        configuration = configparser.ConfigParser(interpolation=None)
        with path.open() as source:
            configuration.read_file(source)
        parse_mkosi_packages(configuration.get("Content", "Packages"))

### check_configuration

In [ ]:
def check_configuration() -> None:
    read_identity(ROOT / "config/identity.json")
    configuration = json.loads((ROOT / "config/build.json").read_text())
    if not re.fullmatch(r"\d{8}T\d{6}Z", configuration["snapshot"]):
        raise ValueError("Invalid Debian snapshot")
    if (
        configuration["release"] != "trixie"
        or configuration["mkosi_package"] != "25.3-7"
    ):
        raise ValueError("Review the toolchain before changing its baseline")

### check_trust

In [ ]:
def check_trust(required: bool) -> None:
    certificate = ROOT / "trust/ipa-ca.crt"
    fingerprint = ROOT / "trust/ipa-ca.sha256"
    if not required and not certificate.exists() and not fingerprint.exists():
        print(
            "IPA CA not configured. Source checks are available; image builds are blocked."
        )
        return
    if not certificate.is_file() or not fingerprint.is_file():
        raise ValueError(
            "Add the real IPA CA with notebooks/trust.ipynb; no placeholder CA is accepted"
        )
    verify_ca(certificate, fingerprint.read_text())

### check_actions

In [ ]:
def check_actions() -> None:
    for workflow in (ROOT / ".github/workflows").glob("*.yml"):
        for action in re.findall(r"uses:\s*([^\s]+)", workflow.read_text()):
            if not re.fullmatch(r"[\w.-]+/[\w./-]+@[a-f0-9]{40}", action):
                raise ValueError("Unpinned action: " + action)

### check_python_syntax

In [ ]:
def check_python_syntax() -> None:
    """Parse host tools and extensionless guest commands without running them."""
    files = [
        *ROOT.glob("lib/**/*.py"),
        *ROOT.glob("tests/*.py"),
    ]
    for command in ROOT.glob("images/*/overlay/usr/local/sbin/*"):
        if command.is_file() and "python" in command.read_text().splitlines()[0]:
            files.append(command)
    for path in files:
        ast.parse(path.read_text(), filename=str(path.relative_to(ROOT)))

### check_notebooks

In [ ]:
def check_notebooks() -> None:
    """Validate all executable notebook cells as Python syntax."""
    required = ("check", "prepare", "make-iso", "artifacts", "footprint", "trust", "seed", "smoke")
    for name in required:
        if not (ROOT / "notebooks" / (name + ".ipynb")).is_file():
            raise ValueError("Missing required build notebook: " + name)
    for path in (ROOT / "notebooks").glob("*.ipynb"):
        notebook = json.loads(path.read_text())
        if notebook.get("nbformat") != 4 or not isinstance(notebook.get("cells"), list):
            raise ValueError("Invalid notebook structure: " + str(path))
        for index, cell in enumerate(notebook["cells"], 1):
            if cell.get("cell_type") == "code":
                ast.parse("".join(cell.get("source", [])), filename=f"{path}:{index}")

### main

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--require-trust", action="store_true")
    arguments = parser.parse_args()
    check_configuration()
    check_mkosi_packages()
    for image in ("server", "installer"):
        check_tree(ROOT / "images" / image / "overlay")
    check_trust(arguments.require_trust)
    check_actions()
    check_python_syntax()
    check_notebooks()
    print("Source checks passed.")

## Execute

The final cell invokes the operation. Any exception means this step failed.

In [ ]:
if __name__ == "__main__":
    import sys
    sys.argv = ['notebooks/check.ipynb', *NOTEBOOK_ARGS]
    try:
        main()
    except (ValueError, OSError, KeyError, SyntaxError) as error:
        sys.exit(str(error))